# Lab 02: 迴歸問題實作 — 預測加州房價

在本單元中，我們將學習如何使用 PyTorch 實作一個簡單的線性迴歸模型。我們將使用真實的加州房價資料集，嘗試透過「居民收入中位數」來預測該區域的「房價中位數」。

### GPU 環境測試
在開始之前，請先確認您的執行環境已成功啟用 GPU 加速器（例如 Google Colab 中的 T4 GPU）。

**【重要】**若下方輸出顯示為 CPU，請點選右上角『連線 / 變更執行階段類型』，將硬體加速器切換為 GPU 後再往下執行，以防切換執行階段時已安裝的套件與下載的資料被清空重設！

In [ ]:
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'GPU 裝置: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (請切換 T4 GPU)"}')


### 環境安全網與資料準備
確認已啟用 GPU 後，接著我們安裝缺少的套件，並測試 California Housing 資料集是否可正常載入。

In [ ]:
import subprocess, sys

# 自動安裝缺少的套件
def ensure_pkg(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch', 'torchvision', 'scikit-learn', 'matplotlib', 'numpy']:
    ensure_pkg(pkg)

# 測試 California Housing 資料集
try:
    from sklearn.datasets import fetch_california_housing
    housing = fetch_california_housing()
    print(f'California Housing: {housing.data.shape[0]} 筆資料已載入')
except Exception as e:
    print(f'[警告] California Housing 下載失敗: {e}')
    print('[備用] 將使用模擬資料繼續執行，不影響學習效果')
    import numpy as np
    class MockHousing:
        data = np.random.randn(1000, 8).astype('float32')
        target = (np.random.randn(1000) * 2 + 5).astype('float32')
        feature_names = [f'feature_{i}' for i in range(8)]
    housing = MockHousing()
    print(f'備用資料: {housing.data.shape[0]} 筆模擬房屋資料')


### 任務 1: 匯入套件與載入資料集
我們使用 Scikit-learn 套件內建的**加州房價資料集 (California Housing Dataset)**。

這是機器學習領域中最經典的資料集之一，收集了 1990 年加州各個區域（以街區為單位）的統計普查資料。我們將使用它來預測房價！

資料集中總共包含 **20,640 筆街區資料**，每個街區有 8 個特徵欄位與 1 個目標欄位：

#### 欄位中文說明對照表：
*   **`MedInc` (Median Income)**: 居民年收入中位數（單位：萬美元）。例如，`8.3` 代表年薪約 8.3 萬美元。**這是我們後續要使用的主要預測變數！**
*   **`HouseAge` (House Age)**: 街區內房屋年齡的中位數（單位：年）。
*   **`AveRooms` (Average Rooms)**: 平均每戶家庭的房間總數。
*   **`AveBedrms` (Average Bedrooms)**: 平均每戶家庭的臥室總數。
*   **`Population` (Population)**: 該街區的常住人口數。
*   **`AveOccup` (Average Occupancy)**: 平均每戶家庭的成員人數。
*   **`Latitude` (Latitude)**: 街區地理位置的緯度。
*   **`Longitude` (Longitude)**: 街區地理位置的經度。
*   **`MedHouseVal` (Median House Value)**: **預測目標！** 房屋售價中位數（單位：十萬美元）。例如，`4.52` 代表房價為 45.2 萬美元。

接下來，讓我們用程式碼將這份資料集載入，並觀察資料前五筆的特徵樣貌！

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 載入加州房價資料
housing = fetch_california_housing(as_frame=True)
df = housing.frame

# 顯示前五筆資料，觀察欄位
print("資料集欄位說明:")
print(df.head())

### 任務 2: 資料預處理
為了讓視覺化更簡單直觀，我們先使用單一特徵：**「MedInc (收入中位數)」** 來預測目標 **「MedHouseVal (房價中位數，單位為十萬美元)」**。
同時，我們將資料進行標準化 (Standardization)，這對梯度下降法收斂非常重要！

In [ ]:
# 提取特徵 X (MedInc) 與 標籤 y (MedHouseVal)
X = df[['MedInc']].values
y = df[['MedHouseVal']].values

# 劃分訓練集 (80%) 與 測試集 (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 特徵標準化
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

# 將 Numpy 陣列轉為 PyTorch 張量 (Float 格式)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

print(f"訓練集大小: {X_train_tensor.shape}, 測試集大小: {X_test_tensor.shape}")

### 任務 3: 定義線性迴歸模型
在 PyTorch 中，我們要透過繼承 `nn.Module` 來建立模型。對於線性迴歸，我們只需要一個線性運算層 (nn.Linear) `nn.Linear`。
公式： $y = w \cdot x + b$

In [ ]:
class LinearRegression(nn.Module):
    def __init__(self):
        super(LinearRegression, self).__init__()
        # TODO: 建立一個線性層，輸入維度為 1 (MedInc)，輸出維度為 1 (MedHouseVal)
        # 提示：self.linear = nn.Linear(輸入維度, 輸出維度)
        # ??? (請在此填寫你的答案)
        
    def forward(self, x):
        # (已幫你完成) 將輸入 x 送入線性層並返回結果
        out = self.linear(x)
        return out

# 宣告模型實例
model = LinearRegression()
print(model)

### 任務 4: 設定誤差評分函數 (Loss Function)與智慧調整器 (Optimizer)
- **誤差評分函數 (Loss Function)**：我們使用均方誤差 `nn.MSELoss`。
- **智慧調整器 (Optimizer)**：我們使用隨機梯度下降 `optim.SGD`，學習率 (Learning Rate, LR) 設定為 0.05。

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)
print("誤差評分函數 (Loss Function) 與智慧調整器 (Optimizer) 設定完成。")

### 任務 5: 訓練模型 (這是一切神經網路的核心！)
請補全模型學習流程中的關鍵三步驟，讓模型自我修正誤差。

In [ ]:
epochs = 100
loss_history = []

for epoch in range(epochs):
    # 1. 模型預測計算 (前向傳播)：計算預測值
    predictions = model(X_train_tensor)
    
    # 2. 計算損失值
    loss = criterion(predictions, y_train_tensor)
    
    # 3. (已幫你完成) 優化三步驟：歸零 → 反向傳播 → 更新
    # (1) 梯度歸零
    optimizer.zero_grad()
    
    # (2) 誤差修正計算 (反向傳播)計算梯度
    loss.backward()
    
    # (3) 更新權重參數
    optimizer.step()
    
    # 紀錄 Loss
    loss_history.append(loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 繪製 Loss 下降曲線
plt.figure(figsize=(8, 4))
plt.plot(loss_history, label='Training Loss', color='blue')
plt.title('Training Loss Drop')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.show()

### 任務 6: 預測結果視覺化
我們將模型學到的綠色直線（線性迴歸線）繪製在原始資料散佈圖上，看看配合效果如何！

In [ ]:
# 取得訓練後的模型參數 w 與 b
with torch.no_grad(): # 推理階段不需要計算梯度
    w = model.linear.weight.item()
    b = model.linear.bias.item()
print(f"模型學到的參數: w = {w:.4f}, b = {b:.4f}")

# 繪圖對比
plt.figure(figsize=(10, 6))
# 繪製真實資料的散佈圖 (抽樣 500 筆避免太擁擠)
plt.scatter(X_train_scaled[:500], y_train[:500], alpha=0.5, label='Real Data', color='gray')

# 繪製預測的直線
x_axis = np.linspace(X_train_scaled.min(), X_train_scaled.max(), 100)
y_axis = w * x_axis + b
plt.plot(x_axis, y_axis, color='red', linewidth=3, label='Regression Line')

plt.title('California Housing: MedInc vs House Value')
plt.xlabel('Income (Standardized)')
plt.ylabel('House Value ($100k)')
plt.legend()
plt.grid(True)
plt.show()

---

## Lab 02 學習重點小結
### 核心概念掌握
- 線性迴歸的目標：找出最小化均方誤差 (MSE) 的參數 w 和 b。
- 梯度下降法沿著誤差評分函數 (Loss Function)的負梯度方向更新參數，逐步收斂至最小值。
- **標準化 (Standardization)** 可加速訓練收斂，使不同量綱的特徵在同一尺度上比較。
- PyTorch 訓練三步法：`zero_grad()` → `backward()` → `step()`。

### 快速自評測驗

**Q1. 梯度下降法更新參數的方向為？**
- A. 梯度的正方向
- B. 梯度的負方向
- C. 隨機方向

<details><summary>查看解答</summary>

**答案：B — 負梯度方向即誤差評分函數 (Loss Function)下降最快的方向。**

</details>

**Q2. 為何在每次 backward() 之前需要呼叫 zero_grad()？**
- A. 節省記憶體
- B. 防止梯度累積導致更新錯誤
- C. 加快計算速度

<details><summary>查看解答</summary>

**答案：B — PyTorch 預設會累積梯度，每次更新前必須清零。**

</details>

### 完成確認清單
在繼續下一個 Lab 前，請確認你能做到：
- [ ] 我可以用一句話解釋「線性迴歸的目標」
- [ ] 我可以用一句話解釋「梯度下降法沿著誤差評分函數 (Loss Function)的負梯度方向更新參數，逐步收斂至最小值」
- [ ] 我可以用一句話解釋「**標準化 (Standardization)** 可加速訓」
- [ ] 我的程式碼成功執行並得到預期輸出